In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder 
from ast import literal_eval
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import optuna
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
import matplotlib.pyplot as plt

In [19]:
train = pd.read_csv('../train_samples.csv')
test = pd.read_csv('../test_samples.csv', lineterminator='\n')
result = pd.read_csv('result.csv')

# Data preprocessing

In [20]:
train = train.drop(['Rated', 'Poster', 'DVD', 'Poster_path', 'Status', 'Tagline', 'Released', 'Actors', 'Plot', 'Original_language', 'Tagline', 'Production_companies', 'Production_countries', 'Overview', 'Title'], axis=1)
test = test.drop(['Rated', 'Poster', 'DVD', 'Poster_path', 'Status', 'Tagline', 'Released', 'Actors', 'Plot', 'Original_language', 'Tagline', 'Production_companies', 'Production_countries', 'Overview', 'Title'], axis=1)

In [21]:
train['Runtime'] = train['Runtime'].fillna('0 min')
train.loc[train['Runtime']=='Released', 'Runtime'] = '0 min'
train.loc[train['Runtime']=='2 h 7 min', 'Runtime'] = '127 min'
train.loc[train['Runtime']=='48S min', 'Runtime'] = '485 min'
train['Runtime'] = train['Runtime'].str[:-4]
train['Runtime'] = pd.to_numeric(train['Runtime'])

test['Runtime'] = test['Runtime'].fillna('0 min')
test.loc[test['Runtime']=='42S min', 'Runtime'] = '425 min'
test.loc[test['Runtime']=='1 h 27 min', 'Runtime'] = '67 min'
test['Runtime'] = test['Runtime'].str[:-4]
test['Runtime'] = pd.to_numeric(test['Runtime'])

In [22]:
director_le = LabelEncoder()
train['Director'] = director_le.fit_transform(train['Director'])
test['Director'] = director_le.fit_transform(test['Director'])

In [23]:
writer_le = LabelEncoder()
train['Writer'] = writer_le.fit_transform(train['Writer'])
test['Writer'] = writer_le.fit_transform(test['Writer'])

In [24]:
train['Genres'] = train['Genres'].fillna('[]')
test['Genres'] = test['Genres'].fillna('[]')
genres = []
for row in train['Genres']:
    row_arr = literal_eval(row)
    for e in row_arr:
        if not e['name'] in genres:
            genres.append(e['name'])
for row in test['Genres']:
    row_arr = literal_eval(row)
    for e in row_arr:
        if not e['name'] in genres:
            genres.append(e['name'])

In [25]:
# for genre in genres:
#     train['Genre_'+genre] = 0
#     test['Genre_'+genre] = 0
# for row_id in range(len(train)):
#     row_arr = literal_eval(train['Genres'][row_id])
#     for genre in genres:
#         for e in row_arr:
#             if genre == e['name']:
#                 train['Genre_'+genre][row_id] = 1
# for row_id in range(len(test)):
#     row_arr = literal_eval(test['Genres'][row_id])
#     for genre in genres:
#         for e in row_arr:
#             if genre == e['name']:
#                 test['Genre_'+genre][row_id] = 1

In [26]:
train = train.drop(['Genres'], axis=1)
test = test.drop(['Genres'], axis=1)

In [27]:
train['Target'] = train['Target'].fillna(0)
train['Target'] = train['Target'] * 100
train['Target'] = train['Target'].astype(int)

In [28]:
train = train.drop(['id'], axis=1)
test_ids = test['id']
test = test.drop(['id'], axis=1)

In [29]:
target_le = LabelEncoder()
# train['Target'] = target_le.fit_transform(train['Target'])

# Learning Classifier

In [31]:
X = train.drop(['Target'], axis=1)
y = train['Target']
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [32]:
target_le = LabelEncoder()
y_train = target_le.fit_transform(y_train)
y_test = target_le.fit_transform(y_test)

In [34]:
model = RandomForestClassifier(max_depth=10,random_state=0)
model.fit(x_train, y_train)
predict = model.predict(x_test)

In [35]:
# prediction = predict
prediction = target_le.inverse_transform(predict)
final_accuracy = accuracy_score(y_test, prediction)
print(final_accuracy)

ValueError: y contains previously unseen labels: [1547 1548]

# Prediction and writing result

In [ ]:
pred = model.predict(test)
pred = target_le.inverse_transform(pred)
result['id'] = test_ids
result['Target'] = pred / 100
result.to_csv('stupid_result1.csv', index=False)

In [ ]:
feature_importance = model.feature_importances_
sorted_idx = np.argsort(feature_importance)
fig = plt.figure(figsize=(12, 12))
bars = plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), np.array(x_test.columns)[sorted_idx])
plt.title('Feature Importance')